# 01 Official numbers audit

Question: are the official figures on household waste collection in Lagos internally consistent, and are they consistent with stated collection capacity?

Inputs: `data/processed/official_stats.csv` (52 records, 2020 to 2026). Source details are in `docs/sources.md`, conflicts in `docs/data_conflicts.md` and assumptions in `docs/assumptions.md`.

In [1]:
import sys
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

import pandas as pd

from lagos_waste import charts
from lagos_waste import official_stats as ost
from lagos_waste.paths import FIGURES

pd.set_option("display.max_colwidth", 80)

## Load and validate

In [2]:
stats = ost.load_official_stats()
problems = ost.validate_official_stats(stats, ost.source_ids_in_log())
print(f"{len(stats)} records, {stats['source_id'].nunique()} sources, validation problems: {problems or 'none'}")
stats.groupby("metric").size().sort_values(ascending=False).head(12)

52 records, 14 sources, validation problems: none


metric
waste_generated_per_day               7
lawma_budget_capital_expenditure      5
lawma_budget_recurrent_expenditure    5
lawma_budget_total_expenditure        5
psp_operators_active                  5
lawma_budget_independent_revenue      4
waste_received_disposal_facilities    3
disposal_truck_trips                  2
waste_received_per_day_computed       2
olusosun_truck_trips                  2
blackspots_cleared                    1
complaints_and_service_requests       1
dtype: int64

## Check 1: does the May 2026 daily average match the monthly total?

LAWMA stated that over 418,500 tonnes were evacuated in May 2026, "representing an average daily 13,200 tonnes" [S01]. May has 31 days [A01].

In [3]:
may_total = 418_500
may_daily = ost.daily_average(may_total, date(2026, 5, 1), date(2026, 5, 31))
stated_daily = 13_200
print(f"Computed daily average: {may_daily:,.0f} t/day")
print(f"Stated daily average:   {stated_daily:,.0f} t/day")
print(f"Stated average x 31:    {stated_daily * 31:,.0f} t (vs {may_total:,} t reported)")
print(f"Gap: {may_daily - stated_daily:,.0f} t/day ({ost.share_of(may_daily - stated_daily, may_daily):.1f}%)")

Computed daily average: 13,500 t/day
Stated daily average:   13,200 t/day
Stated average x 31:    409,200 t (vs 418,500 t reported)
Gap: 300 t/day (2.2%)


The stated daily average does not match the stated monthly total. The gap is small (2.2% of the computed value), but it shows the two figures were not derived from each other with care (C02).

## Check 2: how is tonnage measured?

For 28 July to 3 August 2026, LAWMA reported 18,660 tonnes and 1,866 truck trips, and for 1 August 3,490 tonnes and 349 trips [S02].

In [4]:
for label, tonnes, trips in [("Week 28 Jul to 3 Aug", 18_660, 1_866), ("1 August", 3_490, 349)]:
    print(f"{label}: {tonnes:,} t / {trips:,} trips = {ost.tonnes_per_trip(tonnes, trips):.2f} t per trip")

Week 28 Jul to 3 Aug: 18,660 t / 1,866 trips = 10.00 t per trip
1 August: 3,490 t / 349 trips = 10.00 t per trip


Both ratios are exactly 10.00 tonnes per trip. Real truck loads vary, so an exact ratio on two separate figures indicates that tonnage is estimated as trips multiplied by 10, not weighed at a weighbridge. This rate is adopted as assumption A02.

## Check 3: is reported evacuation consistent with generation and capacity?

In [5]:
week_daily = ost.daily_average(18_660, date(2026, 7, 28), date(2026, 8, 3))
rows = {
    "May 2026 evacuation as share of 15,000 t/day generation (%)": ost.share_of(may_daily, 15_000),
    "May 2026 evacuation as share of 13,000 t/day generation (%)": ost.share_of(may_daily, 13_000),
    "May 2026 evacuation / PSP capacity upper bound 5,000 t/day (x)": may_daily / 5_000,
    "May 2026 evacuation / PSP capacity lower bound 4,000 t/day (x)": may_daily / 4_000,
    "Week of 28 Jul 2026, daily average (t/day)": week_daily,
    "Week of 28 Jul 2026 vs May 2026 daily rate (% lower)": ost.share_of(may_daily - week_daily, may_daily),
    "World Bank 2020 collected share of 13,000 t/day (%)": ost.share_of(4_263, 13_000),
    "May 2026 evacuation / World Bank 2020 collected 4,263 t/day (x)": may_daily / 4_263,
    "May 2026 evacuation / 54% of 13,000 t/day (x)": may_daily / (0.54 * 13_000),
}
pd.Series(rows).round(2).to_frame("value")

,value
"May 2026 evacuation as share of 15,000 t/day generation (%)",90.00
"May 2026 evacuation as share of 13,000 t/day generation (%)",103.85
"May 2026 evacuation / PSP capacity upper bound 5,000 t/day (x)",2.70
"May 2026 evacuation / PSP capacity lower bound 4,000 t/day (x)",3.38
"Week of 28 Jul 2026, daily average (t/day)",2665.71
Week of 28 Jul 2026 vs May 2026 daily rate (% lower),80.25
"World Bank 2020 collected share of 13,000 t/day (%)",32.79
"May 2026 evacuation / World Bank 2020 collected 4,263 t/day (x)",3.17
"May 2026 evacuation / 54% of 13,000 t/day (x)",1.92


Taken at face value, the May 2026 figure means 90% to 104% of all waste generated in Lagos reached disposal facilities. That is 2.7 to 3.4 times what LAWMA's own MD said PSP operators can collect (4,000 to 5,000 t/day, December 2025 [S06]), and 1.9 to 3.2 times the World Bank's estimate of what was collected through approved channels in 2020 to 2021 (4,263 to 7,020 t/day, 33% to 54% of 13,000 t/day [S10]). About two months later, a week reported across disposal facilities averaged 2,666 t/day, 80% below the May rate (C03, C04). These figures cannot all describe the same system at the same level of performance.

## Check 4: truck trips implied by the claims

In [6]:
trip_rows = [
    ("2021, before increase [S07]", 357, False),
    ("2021, after increase [S07]", 850, False),
    ("Week of 28 Jul 2026, daily average [S02]", round(1_866 / 7), False),
    ("Implied by May 2026 figure at 10 t per trip [A02]", may_daily / 10, True),
]
for label, value, _ in trip_rows:
    print(f"{label}: {value:,.0f} trips/day")
print(f"850 trips at 10 t = {ost.trips_to_tonnes(850):,.0f} t/day; at 8 t = {ost.trips_to_tonnes(850, 8):,.0f}; at 12 t = {ost.trips_to_tonnes(850, 12):,.0f}")

2021, before increase [S07]: 357 trips/day
2021, after increase [S07]: 850 trips/day
Week of 28 Jul 2026, daily average [S02]: 267 trips/day
Implied by May 2026 figure at 10 t per trip [A02]: 1,350 trips/day
850 trips at 10 t = 8,500 t/day; at 8 t = 6,800; at 12 t = 10,200


Delivering 13,500 t/day at 10 tonnes per trip requires about 1,350 trips a day, 59% more than the 850 daily trips reported in 2021 and about five times the 267 daily trips reported for late July 2026.

## Check 5: LAWMA approved budget, 2021 to 2026

In [7]:
budget = stats[stats["metric"].str.startswith("lawma_budget_")].copy()
budget["year"] = budget["period_start"].dt.year
wide = budget.pivot_table(index="year", columns="metric", values="value") / 1e9
wide = wide.reindex([2021, 2022, 2023, 2024, 2025, 2026])
wide["recurrent_plus_capital"] = wide["lawma_budget_recurrent_expenditure"] + wide["lawma_budget_capital_expenditure"]
wide.round(2)

metric,lawma_budget_capital_expenditure,lawma_budget_independent_revenue,lawma_budget_recurrent_expenditure,lawma_budget_total_expenditure,recurrent_plus_capital
year,,,,,
2021,1.09,NaN,19.45,20.54,20.54
2022,5.00,1.48,18.22,23.22,23.22
2023,NaN,NaN,NaN,NaN,NaN
2024,3.90,2.30,23.33,27.23,27.23
2025,11.83,3.00,40.38,52.21,52.21
2026,9.37,3.85,41.63,51.00,51.00


In [8]:
tot = wide["lawma_budget_total_expenditure"]
print(f"2024 to 2025 change: {100 * (tot[2025] / tot[2024] - 1):.1f}%")
print(f"2021 to 2026 change: {tot[2026] / tot[2021]:.2f} times (nominal)")
print(f"Capital share 2025: {100 * wide.loc[2025, 'lawma_budget_capital_expenditure'] / tot[2025]:.1f}%; 2026: {100 * wide.loc[2026, 'lawma_budget_capital_expenditure'] / tot[2026]:.1f}%")

2024 to 2025 change: 91.7%
2021 to 2026 change: 2.48 times (nominal)
Capital share 2025: 22.7%; 2026: 18.4%


LAWMA's approved budget nearly doubled between 2024 and 2025 (N27.2bn to N52.2bn) and was 2.48 times its 2021 level in 2026, in nominal terms [A05]. No LAWMA line was located for 2023. Recurrent spending dominates; capital fell from N11.8bn in 2025 to N9.4bn in 2026, while the fleet count reported in 2026 (77 compactors [S01]) was lower than in 2025 (102 [S04]).

## Charts

In [9]:
fig1 = charts.daily_tonnage_chart(
    [
        ("Waste generated, official range 2025-26", 13_000, 15_000, False),
        ("Reported at disposal sites, May 2026", round(may_daily), None, True),
        ("PSP collection capacity, Dec 2025", 4_000, 5_000, False),
        ("Collected via approved channels, 2020", 4_263, None, False),
        ("Reported at disposal sites, 28 Jul-3 Aug 2026", round(week_daily), None, False),
    ],
    "May 2026 disposal figure is 2.7 to 3.4 times stated PSP capacity",
    "Sources: S01, S02, S03, S05, S06, S09, S10. May and July-August values are daily averages of reported totals [A01].",
    FIGURES / "01_daily_tonnage_vs_capacity.png",
)

In [10]:
fig2 = charts.budget_chart(
    ["2021", "2022", "2023", "2024", "2025", "2026"],
    list(wide["lawma_budget_recurrent_expenditure"].where(wide["lawma_budget_recurrent_expenditure"].notna(), None)),
    list(wide["lawma_budget_capital_expenditure"].where(wide["lawma_budget_capital_expenditure"].notna(), None)),
    "LAWMA approved budget nearly doubled in 2025",
    "Sources: S11, S12, S13, S14, S15 (Lagos State appropriation laws, NCoA format). Nominal naira, approved not actual [A05].",
    FIGURES / "01_lawma_budget_2021_2026.png",
)

In [11]:
fig3 = charts.trips_chart(
    trip_rows,
    "May 2026 figure implies about 1,350 truck trips a day",
    "Sources: S02, S07. Hatched bar is derived: 13,500 t/day / 10 t per trip [A02].",
    FIGURES / "01_truck_trips_per_day.png",
)

## Summary

| Check | Result |
|---|---|
| May 2026 daily average | Stated 13,200 t/day; total implies 13,500 t/day (C02) |
| Tonnage measurement | Two figures equal exactly 10 t per trip, consistent with estimation from trip counts |
| Evacuation vs generation | May 2026 implies 90% to 104% of generation evacuated |
| Evacuation vs capacity | 2.7 to 3.4 times the PSP capacity stated by LAWMA in December 2025 |
| Consistency over time | Late July 2026 daily average is 80% below May 2026 (C03) |
| Budget | Approved budget N27.2bn (2024) to N52.2bn (2025) to N51.0bn (2026), nominal |

The official collection figures are not consistent with each other or with LAWMA's own statement of capacity. The resident survey (Phase 2) provides an independent measure of collection frequency.